### Train and test set load & informations

In [2]:
from datasets import load_dataset

ds = load_dataset("axmeu/wiki_fr")
train_texts = ds["train"]["text"][:180_000]
test_texts = ds["test"]["text"]

print(f"Train text length:  {len(train_texts)}")
print(f"Test text length:   {len(test_texts)}")

/home/onyxia/work/.pixi/envs/default/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 5000/5000 [00:00<00:00, 66079.09 examples/s]


Train text length:  180000
Test text length:   5000


In [6]:
import regex
from collections import Counter

def count_sentences(texts):
    sentence_counter = 0
    for text in texts:
        sentences = regex.split(r"[.!?…]+", text)
        sentences = [s for s in sentences if s.strip()]
        sentence_counter += len(sentences)
    return sentence_counter


train_counter = Counter()
for text in train_texts:
    words = regex.findall(r"[a-zA-ZÀ-ÿŒœ]+", text)
    train_counter.update(words)

train_char = sum(len(word) * freq for word, freq in train_counter.items())
train_sentences = count_sentences(train_texts)


test_counter = Counter()
for text in test_texts:
    words = regex.findall(r"[a-zA-ZÀ-ÿŒœ]+", text)
    test_counter.update(words)

test_char = sum(len(word) * freq for word, freq in test_counter.items())
test_sentences = count_sentences(test_texts)


print(f"Total words in train set:      {sum(train_counter.values())}")
print(f"Unique words in train set:     {len(train_counter)}")
print(f"Total characters in train set: {train_char}")
print(f"Total sentences in train set:  {train_sentences}\n")

print(f"Total words in test set:       {sum(test_counter.values())}")
print(f"Unique words in test set:     {len(test_counter)}")
print(f"Total characters in test set: {test_char}")
print(f"Total sentences in test set:  {test_sentences}")

Total words in train set:      234259489
Unique words in train set:     1769884
Total characters in train set: 1152439114
Total sentences in train set:  10154936

Total words in test set:       3203426
Unique words in test set:     161029
Total characters in test set: 15773372
Total sentences in test set:  142327


### Load models, tokenize sequence

In [ ]:
import sys
sys.path.append("..")
from BPE.fast import FastBPE
from WordPiece.fast import FastWordPiece

bpe_fast  = FastBPE.load("../results/models/bpe_fast_v32000_n180000.json")
wp_fast = FastWordPiece.load("../results/models/wp_fast_v32000_n180000.json")

In [ ]:
seq1 = "Une narration @@🫠"
print(f"BPE:  {bpe_fast.tokenize(seq1)}")
print(f"WP:   {wp_fast.tokenize(seq1)}")

seq2 = "La politique de demain"
print(f"\nBPE: {bpe_fast.tokenize(seq2)}")
print(f"WP:  {wp_fast.tokenize(seq2)}")

BPE:  ['Une</w>', 'narration</w>', '@', '@', '🫠']
WP:   ['Une', 'narration', '@', '@', '🫠']

BPE: ['La</w>', 'politique</w>', 'de</w>', 'demain</w>']
WP:  ['L', '##a', 'p', '##o', '##l', '##i', '##t', '##i', '##q', '##u', '##e', 'd', '##e', 'd', '##e', '##m', '##a', '##i', '##n']


In [ ]:
inpt = input("Type a sequence to tokenize: ")

print(f"BPE: {bpe_fast.tokenize(inpt)}")
print(f"WP:  {wp_fast.tokenize(inpt)}")
print(f"\nOriginal: {inpt}")

BPE: ['Ceci</w>', 'est</w>', 'u', 'n', '</w>', 'e', 'x', 'em', 'p', 'le</w>']
WP:  ['C', '##ec', '##i', 'e', '##s', '##t', 'un', 'exempl', '##e']

Original: Ceci est un exemple


### BPE merge rules & WP vocabulary exploration

In [17]:
print("BPE:")
print(f"  Number of merge rules: {len(bpe_fast.merge_rules)}")
print(f"  First merge rules: {list(bpe_fast.merge_rules[:10])}")

BPE:
  Number of merge rules: 31883
  First merge rules: [('e', '</w>'), ('s', '</w>'), ('t', '</w>'), ('e', 's</w>'), ('e', 'n'), ('o', 'n'), ('d', 'e</w>'), ('a', 'n'), ('r', '</w>'), ('a', '</w>')]


In [25]:
import sys
sys.path.append("..")
from src.demo import trace_word_bpe

inpt = input("Type a word to get the rules that tokenizes it: ")
trace_word_bpe(inpt, bpe_fast.merge_rules)


Initial: ['f', 'a', 'u', 'c', 'o', 'n', '</w>']

  Step 1    (rule #5     ): 'o'          + 'n'          → 'on'
  Step 2    (rule #19    ): 'on'         + '</w>'       → 'on</w>'
  Step 3    (rule #21    ): 'a'          + 'u'          → 'au'
  Step 4    (rule #3683  ): 'c'          + 'on</w>'     → 'con</w>'
  Step 5    (rule #15261 ): 'au'         + 'con</w>'    → 'aucon</w>'
  Step 6    (rule #21503 ): 'f'          + 'aucon</w>'  → 'faucon</w>'

Final: ['faucon</w>']


In [19]:
print("WP:")
print(f"  Vocab size: {len(wp_fast.vocab)}")
print(f"  WP vocabulary sample: \n  {list(wp_fast.vocab)[200:210]}\n"
      f"  {list(wp_fast.vocab)[4_000:4_010]}\n"
      f"  {list(wp_fast.vocab)[25_000:25_010]}")

WP:
  Vocab size: 32000
  WP vocabulary sample: 
  ['Bénilde', '##enem', 'Thibier', '##laté', '##oneu', 'dicastèr', '##oplanct', '##haptes', 'situationniste', 'Indifférent']
  ['##hèque', '##kérato', 'Makah', '##oija', 'Turns', '##egraaff', 'Müll', 'stérilisant', '##nvironnemen', 'Ager']
  ['##amgy', '##sentative', 'Latrèch', '##omestiquées', 'subvert', 'dimin', 'ciba', '##âlay', 'dizaines', 'examinons']
